# 02p — InceptionV3 + CBAM Dual-Run Training (3 single-crack classes)

**Project:** UREP 32-0210-250078 | Crack Classification

**Architecture:** InceptionV3 + CBAM (Convolutional Block Attention) — 3-stage transfer learning.

**Classes (3):** debonding, flexural, shear.

## Two training runs in this notebook

1. **Run A — QU only** (3-class subset).
2. **Run B — QU + AutoCAD synthetic with forced mirror** (every synthetic image seen in both orientations every epoch).

Standard augmentation pipeline applies equally to real and synthetic; mirroring is hard-coded on top per the civil-engineering requirement. Test set is identical for both runs (3-class subset of `config.SPLIT_DIR/test/`).

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import torch
import torch.nn as nn

import config
from src.dataset import (
    prepare_dataset, split_dataset, get_dataloaders,
    compute_class_weights, prepare_synthetic_split,
)
from src.model_cbam import InceptionV3CBAM
from src.model import freeze_backbone, unfreeze_from, unfreeze_all
from src.trainer import train_model
from src.evaluation import plot_training_history, evaluate_model
from src.device import print_device_summary, get_device, set_seed

set_seed(config.RANDOM_SEED)

CLASS_NAMES_3 = ["debonding", "flexural", "shear"]
NUM_CLASSES_3 = 3

OUT_QU      = os.path.join(config.OUTPUT_DIR, "cbam_3class_qu")
OUT_QUSYNTH = os.path.join(config.OUTPUT_DIR, "cbam_3class_qu_synth")
for d in (OUT_QU, OUT_QUSYNTH):
    os.makedirs(os.path.join(d, "models"), exist_ok=True)
    os.makedirs(os.path.join(d, "plots"), exist_ok=True)
    os.makedirs(os.path.join(d, "logs"), exist_ok=True)

device_config = print_device_summary()
device = get_device()
STAGE_BATCH = device_config["batch_sizes"]
NUM_WORKERS = device_config["num_workers"]

print(f"\nArchitecture: InceptionV3 + CBAM (3-class)")
print(f"Device:        {device}")
print(f"Classes ({NUM_CLASSES_3}): {CLASS_NAMES_3}")
print(f"OUT_QU:        {OUT_QU}")
print(f"OUT_QUSYNTH:   {OUT_QUSYNTH}")

### Logging helper (used in both runs)

In [ ]:
def log_class_balance(dataset, label):
    """Print per-class counts breaking down real / synth-orig / synth-mirror.

    For Inception/CBAM (PyTorch path): mirrors are virtual entries in
    `dataset.flips` (no `synth_mirror_` on disk).
    """
    print(f"\n{'='*70}\n{label}\n{'='*70}")
    print(f"  Total samples in dataset: {len(dataset):,}")
    print(f"  Classes ({len(dataset.class_counts)}): {dataset.class_names}")
    for cls_idx, cls_name in enumerate(dataset.class_names):
        idx_in_class = [i for i, lbl in enumerate(dataset.labels) if lbl == cls_idx]
        n_total = len(idx_in_class)
        n_mirror = sum(1 for i in idx_in_class if dataset.flips[i])
        n_synth = sum(
            1 for i in idx_in_class
            if not dataset.flips[i]
            and os.path.basename(dataset.file_paths[i]).startswith("synth_")
        )
        n_real = n_total - n_synth - n_mirror
        print(f"  {cls_name:<14} total={n_total:>6,}  real={n_real:>6,}  "
              f"synth={n_synth:>4,}  mirror={n_mirror:>4,}")

### Ensure base split exists

In [ ]:
# Ensure base 6-class split exists (idempotent)
if not os.path.exists(config.SPLIT_DIR) or not os.path.isdir(os.path.join(config.SPLIT_DIR, "train")):
    print("Preparing base dataset from QU structure...")
    prepare_dataset()
    split_dataset()
else:
    print(f"Base split exists: {config.SPLIT_DIR}")

---
# Run A — QU only (3 classes)

### Run A: build dataloaders

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[1],
    img_size=config.IMG_SIZE,
    normalize="imagenet",
    num_workers=NUM_WORKERS,
    class_names=CLASS_NAMES_3,
    mirror_synthetic=False,
)

log_class_balance(train_loader.dataset, "RUN A — QU only — TRAIN")
log_class_balance(val_loader.dataset,   "RUN A — QU only — VAL")
log_class_balance(test_loader.dataset,  "RUN A — QU only — TEST")

### Run A: class weights

In [ ]:
class_weight_dict_a = compute_class_weights(
    split_dir=config.SPLIT_DIR,
    class_names=CLASS_NAMES_3,
    count_synth_mirror=False,
)
class_weight_tensor_a = torch.tensor(
    [class_weight_dict_a[i] for i in range(NUM_CLASSES_3)],
    dtype=torch.float32,
).to(device)
print(f"\nClass weight tensor (Run A): {class_weight_tensor_a}")

### Run A: build model

In [ ]:
model_a = InceptionV3CBAM(num_classes=NUM_CLASSES_3).to(device)
print(f"Total parameters: {sum(p.numel() for p in model_a.parameters()):,}")

### Run A — Stage 1: feature extraction (frozen backbone)

In [ ]:
freeze_backbone(model_a)
criterion_a = nn.CrossEntropyLoss(weight=class_weight_tensor_a)
optimizer_a = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_a.parameters()),
    lr=config.STAGE1_LR,
)
history_a1 = train_model(
    model_a, train_loader, val_loader, optimizer_a, criterion_a, device,
    epochs=config.STAGE1_EPOCHS,
    output_dir=OUT_QU, stage=1, model_name="cbam_3c_qu",
)

### Run A — Stage 2: partial fine-tuning

In [ ]:
if STAGE_BATCH[2] != STAGE_BATCH[1]:
    train_loader, val_loader, _ = get_dataloaders(
        split_dir=config.SPLIT_DIR,
        batch_size=STAGE_BATCH[2], img_size=config.IMG_SIZE,
        normalize="imagenet", num_workers=NUM_WORKERS,
        class_names=CLASS_NAMES_3, mirror_synthetic=False,
    )

unfreeze_from(model_a, "Mixed_7a")
optimizer_a = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_a.parameters()),
    lr=config.STAGE2_LR,
)
history_a2 = train_model(
    model_a, train_loader, val_loader, optimizer_a, criterion_a, device,
    epochs=config.STAGE2_EPOCHS,
    output_dir=OUT_QU, stage=2, model_name="cbam_3c_qu",
)

### Run A — Stage 3: full fine-tuning

In [ ]:
if STAGE_BATCH[3] != STAGE_BATCH[2]:
    train_loader, val_loader, _ = get_dataloaders(
        split_dir=config.SPLIT_DIR,
        batch_size=STAGE_BATCH[3], img_size=config.IMG_SIZE,
        normalize="imagenet", num_workers=NUM_WORKERS,
        class_names=CLASS_NAMES_3, mirror_synthetic=False,
    )

unfreeze_all(model_a)
optimizer_a = torch.optim.AdamW(model_a.parameters(), lr=config.STAGE3_LR)
history_a3 = train_model(
    model_a, train_loader, val_loader, optimizer_a, criterion_a, device,
    epochs=config.STAGE3_EPOCHS,
    output_dir=OUT_QU, stage=3, model_name="cbam_3c_qu",
)

### Run A — training curves, save, evaluate

In [ ]:
plot_training_history(
    [history_a1, history_a2, history_a3],
    output_dir=OUT_QU,
    stage_names=["Feature Extraction", "Partial Fine-Tuning", "Full Fine-Tuning"],
    model_name="cbam_3c_qu",
)
torch.save(model_a.state_dict(), os.path.join(OUT_QU, "models", "best_model.pt"))
print(f"Run A model saved to: {os.path.join(OUT_QU, 'models', 'best_model.pt')}")

metrics_a = evaluate_model(
    model_a, test_loader, device,
    class_names=CLASS_NAMES_3,
    output_dir=OUT_QU,
    model_name="cbam_3c_qu",
)

---
# Run B — QU + AutoCAD synthetic with forced mirror

### Run B: prepare synthetic split (idempotent)

Materializes `data/split_synthetic/` if not already present (original train + synth_*). The synthetic mirror is *not* on disk — it is added on-the-fly via `mirror_synthetic=True`.

In [ ]:
prepare_synthetic_split()
print(f"\nUsing split: {config.SPLIT_SYNTHETIC_DIR}")

### Run B: build dataloaders with `mirror_synthetic=True`

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    split_dir=config.SPLIT_SYNTHETIC_DIR,
    batch_size=STAGE_BATCH[1],
    img_size=config.IMG_SIZE,
    normalize="imagenet",
    num_workers=NUM_WORKERS,
    class_names=CLASS_NAMES_3,
    mirror_synthetic=True,
    test_split_dir=config.SPLIT_DIR,  # val + test from clean QU split
)

log_class_balance(train_loader.dataset, "RUN B — QU+synth+mirror — TRAIN")
log_class_balance(val_loader.dataset,   "RUN B — QU+synth+mirror — VAL  (must match Run A)")
log_class_balance(test_loader.dataset,  "RUN B — QU+synth+mirror — TEST (must match Run A)")

# Sanity: total mirror count should equal the count of synth_* files in the train set
n_mirror = sum(train_loader.dataset.flips)
print(f"\nMirror count (assert): {n_mirror}")
assert n_mirror == 102 + 302 + 402, (
    f"Expected 806 mirrored synth entries (102+302+402), got {n_mirror}"
)
print("OK — every synthetic image has a mirror counterpart in the train set.")

### Run B: class weights (counting mirrors)

In [ ]:
class_weight_dict_b = compute_class_weights(
    split_dir=config.SPLIT_SYNTHETIC_DIR,
    class_names=CLASS_NAMES_3,
    count_synth_mirror=True,
)
class_weight_tensor_b = torch.tensor(
    [class_weight_dict_b[i] for i in range(NUM_CLASSES_3)],
    dtype=torch.float32,
).to(device)
print(f"\nClass weight tensor (Run B): {class_weight_tensor_b}")

### Run B: build fresh model

In [ ]:
model_b = InceptionV3CBAM(num_classes=NUM_CLASSES_3).to(device)
print(f"Total parameters: {sum(p.numel() for p in model_b.parameters()):,}")

### Run B — Stage 1

In [ ]:
freeze_backbone(model_b)
criterion_b = nn.CrossEntropyLoss(weight=class_weight_tensor_b)
optimizer_b = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_b.parameters()),
    lr=config.STAGE1_LR,
)
history_b1 = train_model(
    model_b, train_loader, val_loader, optimizer_b, criterion_b, device,
    epochs=config.STAGE1_EPOCHS,
    output_dir=OUT_QUSYNTH, stage=1, model_name="cbam_3c_qusynth",
)

### Run B — Stage 2

In [ ]:
if STAGE_BATCH[2] != STAGE_BATCH[1]:
    train_loader, val_loader, _ = get_dataloaders(
        split_dir=config.SPLIT_SYNTHETIC_DIR,
        batch_size=STAGE_BATCH[2], img_size=config.IMG_SIZE,
        normalize="imagenet", num_workers=NUM_WORKERS,
        class_names=CLASS_NAMES_3, mirror_synthetic=True,
        test_split_dir=config.SPLIT_DIR,
    )

unfreeze_from(model_b, "Mixed_7a")
optimizer_b = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_b.parameters()),
    lr=config.STAGE2_LR,
)
history_b2 = train_model(
    model_b, train_loader, val_loader, optimizer_b, criterion_b, device,
    epochs=config.STAGE2_EPOCHS,
    output_dir=OUT_QUSYNTH, stage=2, model_name="cbam_3c_qusynth",
)

### Run B — Stage 3

In [ ]:
if STAGE_BATCH[3] != STAGE_BATCH[2]:
    train_loader, val_loader, _ = get_dataloaders(
        split_dir=config.SPLIT_SYNTHETIC_DIR,
        batch_size=STAGE_BATCH[3], img_size=config.IMG_SIZE,
        normalize="imagenet", num_workers=NUM_WORKERS,
        class_names=CLASS_NAMES_3, mirror_synthetic=True,
        test_split_dir=config.SPLIT_DIR,
    )

unfreeze_all(model_b)
optimizer_b = torch.optim.AdamW(model_b.parameters(), lr=config.STAGE3_LR)
history_b3 = train_model(
    model_b, train_loader, val_loader, optimizer_b, criterion_b, device,
    epochs=config.STAGE3_EPOCHS,
    output_dir=OUT_QUSYNTH, stage=3, model_name="cbam_3c_qusynth",
)

### Run B — training curves, save, evaluate

In [ ]:
plot_training_history(
    [history_b1, history_b2, history_b3],
    output_dir=OUT_QUSYNTH,
    stage_names=["Feature Extraction", "Partial Fine-Tuning", "Full Fine-Tuning"],
    model_name="cbam_3c_qusynth",
)
torch.save(model_b.state_dict(), os.path.join(OUT_QUSYNTH, "models", "best_model.pt"))
print(f"Run B model saved to: {os.path.join(OUT_QUSYNTH, 'models', 'best_model.pt')}")

# Evaluate on the same clean 3-class test set used by Run A
metrics_b = evaluate_model(
    model_b, test_loader, device,
    class_names=CLASS_NAMES_3,
    output_dir=OUT_QUSYNTH,
    model_name="cbam_3c_qusynth",
)

---
# Comparison: Run A vs Run B

In [ ]:
print(f"\n{'='*78}")
print(f"INCEPTIONV3 + CBAM — 3-CLASS COMPARISON  (test set: {len(test_loader.dataset):,} images)")
print(f"{'='*78}")
print(f"  {'Metric':<24} {'Run A: QU only':>18} {'Run B: QU+synth+mir':>22}")
print(f"  {'-'*64}")
for key in ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted", "mean_iou"]:
    print(f"  {key:<24} {metrics_a[key]:>18.4f} {metrics_b[key]:>22.4f}")
print(f"  {'-'*64}")
print(f"  Per-class IoU:")
for cls in CLASS_NAMES_3:
    a = metrics_a["iou_per_class"][cls]
    b = metrics_b["iou_per_class"][cls]
    delta = b - a
    print(f"    {cls:<14} {a:>18.4f} {b:>22.4f}   (Δ={delta:+.4f})")
print(f"{'='*78}")